In [7]:
import yfinance as yf, numpy as np, pandas as pd
P={'AGL':('AGL.JO','AAL.L','GBPZAR=X'),'CFR':('CFR.JO','CFR.SW','CHFZAR=X'),'BTI':('BTI.JO','BATS.L','GBPZAR=X'),
   'GLN':('GLN.JO','GLEN.L','GBPZAR=X'),'MNP':('MNP.JO','MNDI.L','GBPZAR=X'),'BHG':('BHG.JO','BHP.AX','AUDZAR=X'),
   'S32':('S32.JO','S32.AX','AUDZAR=X'),'PRX':('PRX.JO','PRX.AS','EURZAR=X'),'ANG':('ANG.JO','AU','USDZAR=X')}
t=sorted({x for v in P.values() for x in v})
R=np.log(yf.download(t,start='2003-10-01',auto_adjust=True,progress=False)['Close'].resample('ME').last()).diff()
E=pd.read_csv('data/thesis_data.csv',index_col=0,parse_dates=True)
em=E['MXEF Index_Log_Diff'].reindex(R.index,method='nearest'); mm=E['MXWO0MM Index_Log_Diff'].reindex(R.index,method='nearest')
rows=[]
for k,(j,f,fx) in P.items():
    d=pd.concat([R[j],R[fx],em,mm],axis=1).dropna(); d.columns=['j','z','em','mm']
    if len(d)<104: rows.append((k,len(d),None,None)); continue
    X=np.c_[np.ones(len(d)),d[['em','mm']]]
    rj=d.j-X@np.linalg.lstsq(X,d.j,rcond=None)[0]; rz=d.z-X@np.linalg.lstsq(X,d.z,rcond=None)[0]
    rows.append((k,len(d),round(np.polyfit(d.z,d.j,1)[0],2),round(np.polyfit(rz,rj,1)[0],2)))
pd.DataFrame(rows,columns=['name','weeks','raw slope','partialled slope'])

,name,weeks,raw slope,partialled slope
0,AGL,273,-0.22,-0.22
1,CFR,41,NaN,NaN
2,BTI,273,0.45,0.48
3,GLN,154,-0.34,-0.38
4,MNP,231,0.33,0.32
5,BHG,56,NaN,NaN
6,S32,136,0.49,0.53
7,PRX,84,NaN,NaN
8,ANG,273,0.07,0.09


In [8]:
import yfinance as yf, numpy as np, pandas as pd
P={'AGL':('AGL.JO','AAL.L','GBPZAR=X'),'CFR':('CFR.JO','CFR.SW','CHFZAR=X'),'BTI':('BTI.JO','BATS.L','GBPZAR=X'),
   'GLN':('GLN.JO','GLEN.L','GBPZAR=X'),'MNP':('MNP.JO','MNDI.L','GBPZAR=X'),'BHG':('BHG.JO','BHP.AX','AUDZAR=X'),
   'S32':('S32.JO','S32.AX','AUDZAR=X'),'PRX':('PRX.JO','PRX.AS','EURZAR=X'),'ANG':('ANG.JO','AU','USDZAR=X')}
t=sorted({x for v in P.values() for x in v})
px=yf.download(t,start='2003-10-01',auto_adjust=True,progress=False)['Close']
E=pd.read_csv('data/thesis_data.csv',index_col=0,parse_dates=True)
out={}
for freq,label in [('W-FRI','weekly'),('ME','monthly')]:
    R=np.log(px.resample(freq).last()).diff()
    em=E['MXEF Index_Log_Diff'].resample(freq).sum(min_count=1).reindex(R.index)   # sum weekly log-diffs within the period
    mm=E['MXWO0MM Index_Log_Diff'].resample(freq).sum(min_count=1).reindex(R.index)
    for k,(j,f,fx) in P.items():
        d=pd.concat([R[j],R[fx],em,mm],axis=1).dropna(); d.columns=['j','z','em','mm']
        if k=='ANG': d=d.loc['2023-10-01':]          # Johannesburg was the primary listing before then
        if len(d)<36: out[(k,label)]=(len(d),None,None); continue
        X=np.c_[np.ones(len(d)),d[['em','mm']]]
        rj=d.j-X@np.linalg.lstsq(X,d.j,rcond=None)[0]; rz=d.z-X@np.linalg.lstsq(X,d.z,rcond=None)[0]
        out[(k,label)]=(len(d),round(np.polyfit(d.z,d.j,1)[0],2),round(np.polyfit(rz,rj,1)[0],2))
T=pd.DataFrame(out,index=['n','raw','partialled']).T.unstack()
T

n             raw        partialled       
    monthly  weekly monthly weekly    monthly weekly
AGL   273.0  1187.0   -0.22  -0.18       0.42   0.54
ANG    36.0   153.0   -1.41  -0.01       0.54   0.09
BHG    56.0   239.0    0.46   0.21       0.71   0.59
BTI   273.0  1187.0    0.45   0.29       0.45   0.43
CFR    41.0   176.0    0.12   0.22       0.31   0.75
GLN   154.0   668.0   -0.35  -0.03       0.37   0.59
MNP   231.0  1001.0    0.33   0.12       0.70   0.57
PRX    84.0   364.0    0.39   0.07       1.02   0.67
S32   136.0   590.0    0.49   0.38       0.89   0.63

In [9]:
out={}
for freq,label in [('W-FRI','weekly'),('ME','monthly')]:
    R=np.log(px.resample(freq).last()).diff()
    for k,(j,f,fx) in P.items():
        d=pd.concat([R[j],R[f],R[fx]],axis=1).dropna(); d.columns=['j','f','x']
        if k=='ANG': d=d.loc['2023-10-01':]
        if len(d)<36: continue
        out[(k,label)]=round(np.polyfit(d.f+d.x,d.j,1)[0],2)
pd.Series(out).unstack()

,monthly,weekly
AGL,1.00,0.92
ANG,0.99,0.72
BHG,0.90,0.94
BTI,0.87,0.74
CFR,1.03,0.93
GLN,1.03,0.95
MNP,0.96,0.89
PRX,0.99,0.96
S32,0.98,0.94
